In [1]:
import sys
print(sys.executable)

/Users/nikos/anaconda3/envs/ham10000_ag_legacy/bin/python


In [2]:
from autogluon.multimodal import MultiModalPredictor
print("AutoGluon works")

/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


AutoGluon works


In [3]:
from autogluon.core.utils.loaders import load_zip

download_dir = "./ag_multimodal_tutorial"
zip_file = "https://automl-mm-bench.s3.amazonaws.com/petfinder_for_tutorial.zip"

load_zip.unzip(zip_file, unzip_dir=download_dir)

In [4]:
import numpy as np
import pandas as pd
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(123)

In [5]:
!python3 -m pip install openpyxl

In [6]:
!mkdir -p price_of_books
!wget https://automl-mm-bench.s3.amazonaws.com/machine_hack_competitions/predict_the_price_of_books/Data.zip -O price_of_books/Data.zip
!cd price_of_books && unzip -o Data.zip
!ls price_of_books/Participants_Data

--2026-07-15 09:55:07--  https://automl-mm-bench.s3.amazonaws.com/machine_hack_competitions/predict_the_price_of_books/Data.zip
Aufl"osen des Hostnamens automl-mm-bench.s3.amazonaws.com (automl-mm-bench.s3.amazonaws.com)... 52.216.59.121, 16.15.214.133, 52.216.210.73, ...
Verbindungsaufbau zu automl-mm-bench.s3.amazonaws.com (automl-mm-bench.s3.amazonaws.com)|52.216.59.121|:443 ... verbunden.
HTTP-Anforderung gesendet, auf Antwort wird gewartet ... 200 OK
L"ange: 3521673 (3.4M) [application/zip]
Wird in >>price_of_books/Data.zip<< gespeichert.

price_of_books/Data 100%[===================>]   3.36M  1.44MB/s    in 2.3s    

2026-07-15 09:55:11 (1.44 MB/s) - >>price_of_books/Data.zip<< gespeichert [3521673/3521673]

Archive:  Data.zip
  inflating: Participants_Data/Data_Test.xlsx  
  inflating: Participants_Data/Data_Train.xlsx  
  inflating: Participants_Data/Sample_Submission.xlsx  
Data_Test.xlsx         Data_Train.xlsx        Sample_Submission.xlsx


In [7]:
train_df = pd.read_excel(os.path.join('price_of_books', 'Participants_Data', 'Data_Train.xlsx'), engine='openpyxl')
train_df.head()

,Title,Author,Edition,Reviews,Ratings,Synopsis,Genre,BookCategory,Price
0,The Prisoner's Gold (The Hunters 3),Chris Kuzneski,"Paperback,– 10 Mar 2016",4.0 out of 5 stars,8 customer reviews,THE HUNTERS return in their third brilliant no...,Action & Adventure (Books),Action & Adventure,220.00
1,Guru Dutt: A Tragedy in Three Acts,Arun Khopkar,"Paperback,– 7 Nov 2012",3.9 out of 5 stars,14 customer reviews,A layered portrait of a troubled genius for wh...,Cinema & Broadcast (Books),"Biographies, Diaries & True Accounts",202.93
2,Leviathan (Penguin Classics),Thomas Hobbes,"Paperback,– 25 Feb 1982",4.8 out of 5 stars,6 customer reviews,"""During the time men live without a common Pow...",International Relations,Humour,299.00
3,A Pocket Full of Rye (Miss Marple),Agatha Christie,"Paperback,– 5 Oct 2017",4.1 out of 5 stars,13 customer reviews,A handful of grain is found in the pocket of a...,Contemporary Fiction (Books),"Crime, Thriller & Mystery",180.00
4,LIFE 70 Years of Extraordinary Photography,Editors of Life,"Hardcover,– 10 Oct 2006",5.0 out of 5 stars,1 customer review,"For seven decades, ""Life"" has been thrilling t...",Photography Textbooks,"Arts, Film & Photography",965.62


In [8]:
train_df.shape

(6237, 9)

In [9]:
## Removes out of 5 stars in Reviews and customer reviews in Ratings.
def preprocess(df):
    df = df.copy(deep=True)
    df.loc[:, 'Reviews'] = pd.to_numeric(df['Reviews'].apply(lambda ele: ele[:-len(' out of 5 stars')]))
    df.loc[:, 'Ratings'] = pd.to_numeric(df['Ratings'].apply(lambda ele: ele.replace(',', '')[:-len(' customer reviews')]))
    df.loc[:, 'Price'] = np.log(df['Price'] + 1)
    return df

In [10]:
train_df = pd.read_excel(
    os.path.join(
        "price_of_books",
        "Participants_Data",
        "Data_Train.xlsx"
    ),
    engine="openpyxl"
)
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=123,
    shuffle=True
)

print("Training:", train_data.shape)
print("Testing:", test_data.shape)

Training: (5613, 9)
Testing: (624, 9)


In [11]:
from autogluon.multimodal import MultiModalPredictor
import uuid

time_limit = 15 * 60  # set to larger value in your applications
model_path = f"./tmp/{uuid.uuid4().hex}-automm_text_book_price_prediction"
predictor = MultiModalPredictor(label='Price', path=model_path)
predictor.fit(train_data, time_limit=time_limit)

=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.18
Operating System:   Darwin
Platform Machine:   x86_64
Platform Version:   Darwin Kernel Version 23.6.0: Thu Sep 12 23:34:49 PDT 2024; root:xnu-10063.141.1.701.1~1/RELEASE_X86_64
CPU Count:          8
Pytorch Version:    2.2.2
CUDA Version:       CUDA is not available
Memory Avail:       7.51 GB / 16.00 GB (47.0%)
Disk Space Avail:   84.03 GB / 465.63 GB (18.0%)
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (14100.0, 25.0, 562.67764, 696.56138)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])

AutoMM starts to create your model. ✨✨✨

To track the learning progress, you can open a terminal and 

Sanity Checking: |                                                                                                                                                                   | 0/? [00:00<?, ?it/s]

/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Epoch 0:  10%|███████████████▌                                                                                                                                          | 64/632 [14:35<2:09:33,  0.07it/s]

...

Epoch 0:  10%|███████████████▊                                                                                                                                          | 65/632 [15:01<2:11:02,  0.07it/s]
Validation: |                                                                                                                                                                        | 0/? [00:00<?, ?it/s]
Validation: |                                                                                                                                                                        | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  49%|█████████████████████████████████████████████████████████████████████▌                                                                       | 35/71 [02:31<02:35,  0.23it/s]


Epoch 0:  10%|███████████████▊                                                                                                                                          | 65/632 [20:21<2:57:32,  0.05it/s]

Epoch 0, global step 4: 'val_rmse' reached 1.47226 (best 1.47226), saving model to '/Users/nikos/Documents/Master/Thesis/Software/Multimodal_CP/tmp/eaf6ae50d2114e298c76ba50ebb8ec86-automm_text_book_price_prediction/epoch=0-step=4.ckpt' as top 3


Epoch 0:  10%|███████████████▊                                                                                                                                          | 65/632 [20:27<2:58:25,  0.05it/s]


AutoMM has created your model. 🎉🎉🎉

To load the model, use the code below:
    ```python
    from autogluon.multimodal import MultiModalPredictor
    predictor = MultiModalPredictor.load("/Users/nikos/Documents/Master/Thesis/Software/Multimodal_CP/tmp/eaf6ae50d2114e298c76ba50ebb8ec86-automm_text_book_price_prediction")
    ```

If you are not satisfied with the model, try to increase the training time, 
adjust the hyperparameters (https://auto.gluon.ai/stable/tutorials/multimodal/advanced_topics/customization.html),
or post issues on GitHub (https://github.com/autogluon/autogluon/issues).




In [12]:
predictions = predictor.predict(test_data)
print('Predictions:')
print('------------')
print(np.exp(predictions) - 1)
print()
print('True Value:')
print('------------')
print(np.exp(test_data['Price']) - 1)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/nikos/anaconda3/envs/ham10000_ag_legacy/lib/python3.10/site-packages/autogluon/multimodal/data/templates.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Predicting DataLoader 0: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [07:17<00:00,  0.05it/s]
Predictions:
------------
3861    inf
3340    inf
5785    inf
1271    inf
4528    inf
       ... 
5732    inf
3400    inf
2639    inf
4340    inf
2357    inf
Name: Price, Length: 624, dtype: float32

True Value:
------------
3861              inf
3340     9.253782e+29
5785     2.362183e+93
1271    1.620900e+200
4528     5.685720e+24
            ...      
5732     1.373383e+32
3400    1.920871e+173
2639              inf
4340    1.824082e+193
2357     2.268329e+70
Name: Price, Length: 624, dtype: float64
